# sEEG EDF file viewer (Google Colab)

This notebook loads `dataset/sEEG-HFOs-8.edf`, displays its recording and channel metadata, provides an interactive channel/time viewer, and plots a power spectrum. Run the cells from top to bottom in Google Colab.

> Colab runs Linux, so a local path written as `\dataset\sEEG-HFOs-8.edf` should be uploaded or mounted at `/content/dataset/sEEG-HFOs-8.edf`.

In [ ]:
# Install the EDF reader and interactive controls in the Colab runtime.
%pip install -q mne edfio ipywidgets

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from IPython.display import display

mne.set_log_level("WARNING")
plt.rcParams.update({"figure.figsize": (14, 7), "axes.grid": True})

## Locate the EDF file

The first path below is the expected Colab location. If the dataset is on your computer instead, open Colab's **Files** panel, create a `dataset` folder, and upload the EDF file into it. Alternatively, uncomment and run the upload snippet; uploaded files are temporary and disappear when the runtime resets.

In [ ]:
# Optional browser upload (uncomment only if the file is not in /content/dataset).
# from google.colab import files
# uploaded = files.upload()
# Path("/content/dataset").mkdir(parents=True, exist_ok=True)
# uploaded_name = next(iter(uploaded))
# Path(uploaded_name).replace(Path("/content/dataset/sEEG-HFOs-8.edf"))

In [ ]:
candidates = [
    Path("/content/dataset/sEEG-HFOs-8.edf"),  # Google Colab
    Path("dataset/sEEG-HFOs-8.edf"),           # local Jupyter / repository
]
edf_path = next((path for path in candidates if path.is_file()), None)
if edf_path is None:
    searched = "\n".join(f"  - {path.resolve()}" for path in candidates)
    raise FileNotFoundError(
        "sEEG-HFOs-8.edf was not found. Upload it with the cell above. "
        f"Searched:\n{searched}"
    )

print(f"Loading: {edf_path.resolve()}")
raw = mne.io.read_raw_edf(edf_path, preload=False, verbose="WARNING")
print("EDF loaded successfully.")

## Recording and channel information

In [ ]:
duration_s = raw.n_times / raw.info["sfreq"]
summary = pd.Series({
    "File": str(edf_path),
    "Channels": raw.info["nchan"],
    "Sampling frequency (Hz)": raw.info["sfreq"],
    "Samples per channel": raw.n_times,
    "Duration (seconds)": duration_s,
    "Duration (HH:MM:SS)": pd.to_timedelta(duration_s, unit="s"),
    "Start time": raw.info["meas_date"],
})
display(summary.to_frame("Value"))

channel_table = pd.DataFrame({
    "index": np.arange(len(raw.ch_names)),
    "name": raw.ch_names,
    "type": raw.get_channel_types(),
})
display(channel_table)

## Interactive signal viewer

Select one or more channels and move the time slider. Signals are displayed in microvolts and vertically separated. The viewer reads only the selected segment, which keeps large EDF files manageable.

In [ ]:
default_channels = tuple(raw.ch_names[: min(8, len(raw.ch_names))])
channel_picker = widgets.SelectMultiple(
    options=raw.ch_names, value=default_channels, description="Channels",
    rows=min(12, len(raw.ch_names)), layout=widgets.Layout(width="45%"),
)
window_picker = widgets.Dropdown(
    options=[1.0, 2.0, 5.0, 10.0, 20.0, 30.0], value=10.0,
    description="Window (s)",
)
start_picker = widgets.FloatSlider(
    value=0.0, min=0.0, max=max(0.0, duration_s - window_picker.value),
    step=max(0.1, min(1.0, duration_s / 1000)), description="Start (s)",
    continuous_update=False, readout_format=".1f",
    layout=widgets.Layout(width="90%"),
)

def update_slider_range(change=None):
    start_picker.max = max(0.0, duration_s - window_picker.value)
    start_picker.value = min(start_picker.value, start_picker.max)

window_picker.observe(update_slider_range, names="value")

def plot_segment(channels, start, window):
    if not channels:
        print("Select at least one channel.")
        return
    stop = min(start + window, duration_s)
    data, times = raw.get_data(picks=list(channels), tmin=start, tmax=stop, return_times=True)
    data_uv = data * 1e6
    robust_span = np.nanpercentile(np.abs(data_uv), 95)
    spacing = max(1.0, 2.5 * robust_span)
    offsets = np.arange(len(channels))[::-1] * spacing

    fig, ax = plt.subplots(figsize=(15, max(4, 0.55 * len(channels))))
    for signal, offset, name in zip(data_uv, offsets, channels):
        ax.plot(times, signal + offset, linewidth=0.7, label=name)
    ax.set_yticks(offsets, labels=channels)
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Channel (signals offset; amplitude in µV)")
    ax.set_title(f"sEEG: {start:.2f}–{stop:.2f} s")
    ax.margins(x=0)
    plt.show()

controls = {"channels": channel_picker, "start": start_picker, "window": window_picker}
viewer = widgets.interactive_output(plot_segment, controls)
display(widgets.VBox([channel_picker, window_picker, start_picker]), viewer)

## Power spectral density

Run this cell after choosing channels and a time window above. The upper frequency is limited by both 250 Hz and the recording's Nyquist frequency.

In [ ]:
selected_channels = list(channel_picker.value)
if not selected_channels:
    raise ValueError("Select at least one channel in the viewer first.")

tmin = start_picker.value
tmax = min(tmin + window_picker.value, duration_s)
fmax = min(250.0, raw.info["sfreq"] / 2.0)
spectrum = raw.compute_psd(
    method="welch", picks=selected_channels, tmin=tmin, tmax=tmax,
    fmin=0.5, fmax=fmax, verbose="WARNING",
)
spectrum.plot(average=False, amplitude=False, picks=selected_channels)
plt.show()

## Optional: export a selected segment

The following example creates a new in-memory MNE object containing the current selection. Uncomment the last line to save it in MNE's lossless FIF format.

In [ ]:
segment = raw.copy().pick(list(channel_picker.value)).crop(
    tmin=start_picker.value,
    tmax=min(start_picker.value + window_picker.value, duration_s),
).load_data()
print(segment)
# segment.save("selected_seeg_segment_raw.fif", overwrite=True)